# 1 Парсинг

## 1.1 Мосбиржа 

In [38]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def parse_moex_russian():
    """Парсинг российских акций с Московской биржи"""
    
    # Российские тикеры (Yandex теперь под тикером YNDX)
    tickers = ['SBER', 'GAZP', 'LKOH', 'ROSN', 'YNDX', 'VTBR', 'TATN', 'GMKN']
    
    all_data = {}
    
    for ticker in tickers:
        print(f"Получаем данные для {ticker}...")
        
        url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"
        
        # Берем данные за последние 6 месяцев для теста
        params = {
            'from': '2023-07-01',
            'till': '2024-01-15',
            'interval': 24,  # дневные данные
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            data = response.json()
            
            if 'candles' in data and data['candles']['data']:
                candles = data['candles']['data']
                
                df = pd.DataFrame(candles, columns=[
                    'open', 'close', 'high', 'low', 'value', 'volume', 'begin', 'end'
                ])
                
                df['date'] = pd.to_datetime(df['begin'])
                df['ticker'] = ticker
                df = df[['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']]
                df = df.sort_values('date').reset_index(drop=True)
                
                all_data[ticker] = df
                print(f"  ✓ {ticker}: {len(df)} записей")
            else:
                print(f"  ✗ {ticker}: нет данных")
                
        except Exception as e:
            print(f"  ✗ {ticker}: ошибка - {e}")
    
    return all_data

# Использование
moex_data = parse_moex_russian()

# Посмотрим на данные по Сберу
if 'SBER' in moex_data:
    print("\nДанные SBER:")
    print(moex_data['SBER'].head())
    print(f"Всего записей: {len(moex_data['SBER'])}")

Получаем данные для SBER...
  ✓ SBER: 139 записей
Получаем данные для GAZP...
  ✓ GAZP: 139 записей
Получаем данные для LKOH...
  ✓ LKOH: 139 записей
Получаем данные для ROSN...
  ✓ ROSN: 139 записей
Получаем данные для YNDX...
  ✓ YNDX: 139 записей
Получаем данные для VTBR...
  ✓ VTBR: 139 записей
Получаем данные для TATN...
  ✓ TATN: 139 записей
Получаем данные для GMKN...
  ✓ GMKN: 139 записей

Данные SBER:
        date ticker    open    high     low   close    volume
0 2023-07-03   SBER  240.00  244.56  239.15  243.33  47333310
1 2023-07-04   SBER  243.40  243.78  240.61  240.70  41561500
2 2023-07-05   SBER  241.49  241.90  240.24  240.95  19972080
3 2023-07-06   SBER  241.47  242.47  240.71  240.96  23260340
4 2023-07-07   SBER  241.49  243.75  240.70  243.67  30275210
Всего записей: 139


In [50]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def get_moex_data_reliable(ticker='SBER', days=180):
    """Надежный способ получения данных через MOEX API"""
    
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    url = f"https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/{ticker}.json"
    
    params = {
        'from': start_date.strftime('%Y-%m-%d'),
        'till': end_date.strftime('%Y-%m-%d'),
        'start': 0
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        if 'history' in data and data['history']['data']:
            history_data = data['history']['data']
            
            # Создаем DataFrame
            df = pd.DataFrame(history_data)
            
            # Отладочная информация
            print(f"Доступные колонки для {ticker}: {list(df.columns)}")
            
            # Сопоставление колонок - используем те, что есть в ответе
            column_mapping = {
                'TRADEDATE': 'date',
                'OPEN': 'open',
                'HIGH': 'high', 
                'LOW': 'low',
                'CLOSE': 'close',
                'VOLUME': 'volume'
            }
            
            # Выбираем только существующие колонки
            available_columns = {}
            for moex_col, our_col in column_mapping.items():
                if moex_col in df.columns:
                    available_columns[moex_col] = our_col
            
            if not available_columns:
                print(f"✗ {ticker}: нет подходящих колонок в ответе")
                return pd.DataFrame()
            
            # Переименовываем колонки
            df = df[list(available_columns.keys())].rename(columns=available_columns)
            
            # Преобразуем дату
            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'])
            else:
                print(f"✗ {ticker}: нет колонки с датой")
                return pd.DataFrame()
            
            df['ticker'] = ticker
            df = df.sort_values('date').reset_index(drop=True)
            
            print(f"✓ {ticker}: {len(df)} записей")
            return df
        else:
            print(f"✗ {ticker}: нет исторических данных")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"✗ {ticker}: ошибка - {e}")
        return pd.DataFrame()

# Тестируем на нескольких тикерах
tickers = ['SBER', 'GAZP', 'LKOH', 'VTBR']
reliable_data = {}

print("=== ПАРСИНГ ДАННЫХ С MOEX ===")
for ticker in tickers:
    df = get_moex_data_reliable(ticker, days=30)  # начнем с 30 дней для теста
    if not df.empty:
        reliable_data[ticker] = df

# Показываем результат
if reliable_data:
    first_ticker = list(reliable_data.keys())[0]
    print(f"\n✅ Успешно получены данные для: {list(reliable_data.keys())}")
    print(f"\nПример данных для {first_ticker}:")
    print(reliable_data[first_ticker].head())
else:
    print("\n❌ Не удалось получить данные ни по одному тикеру")

=== ПАРСИНГ ДАННЫХ С MOEX ===
Доступные колонки для SBER: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
✗ SBER: нет подходящих колонок в ответе
Доступные колонки для GAZP: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
✗ GAZP: нет подходящих колонок в ответе
Доступные колонки для LKOH: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
✗ LKOH: нет подходящих колонок в ответе
Доступные колонки для VTBR: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
✗ VTBR: нет подходящих колонок в ответе

❌ Не удалось получить данные ни по одному тикеру


## 1.2 Finam 

In [46]:
import yfinance as yf
import pandas as pd

def parse_yfinance_russian():
    """Парсинг российских акций через yfinance с правильными тикерами"""
    
    # Правильные тикеры для yfinance
    tickers = {
        'SBER.ME': 'Sberbank',
        'GAZP.ME': 'Gazprom', 
        'LKOH.ME': 'Lukoil',
        'ROSN.ME': 'Rosneft',
        'VTBR.ME': 'VTB',
        'GMKN.ME': 'Norilsk Nickel'
    }
    
    all_data = {}
    
    for ticker, name in tickers.items():
        print(f"Получаем {name} ({ticker})...")
        
        try:
            # Скачиваем данные
            stock = yf.Ticker(ticker)
            df = stock.history(period="1y")  # данные за год
            
            if not df.empty:
                df = df.reset_index()[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
                df = df.rename(columns={
                    'Date': 'date',
                    'Open': 'open', 
                    'High': 'high',
                    'Low': 'low', 
                    'Close': 'close',
                    'Volume': 'volume'
                })
                df['ticker'] = ticker.replace('.ME', '')
                
                df = df.sort_values('date').reset_index(drop=True)
                all_data[ticker.replace('.ME', '')] = df
                print(f"  ✓ {ticker}: {len(df)} записей")
            else:
                print(f"  ✗ {ticker}: нет данных")
                
        except Exception as e:
            print(f"  ✗ {ticker}: ошибка - {e}")
    
    return all_data

# Использование (требует: pip install yfinance)
yfinance_data = parse_yfinance_russian()

if 'SBER' in yfinance_data:
    print("\nДанные SBER через yfinance:")
    print(yfinance_data['SBER'].head())

Получаем Sberbank (SBER.ME)...


$SBER.ME: possibly delisted; no price data found  (period=1y)


  ✗ SBER.ME: нет данных
Получаем Gazprom (GAZP.ME)...


$GAZP.ME: possibly delisted; no price data found  (period=1y)


  ✗ GAZP.ME: нет данных
Получаем Lukoil (LKOH.ME)...


...

  ✗ ROSN.ME: нет данных
Получаем VTB (VTBR.ME)...


$VTBR.ME: possibly delisted; no price data found  (period=1y)


  ✗ VTBR.ME: нет данных
Получаем Norilsk Nickel (GMKN.ME)...


$GMKN.ME: possibly delisted; no price data found  (period=1y)


  ✗ GMKN.ME: нет данных


## 1.3  Investing.com

In [90]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

def parse_investing_yndx():
    """Парсинг котировок Яндекс через yfinance"""
    
    # Для российских акций в yfinance используется .ME
    ticker = "TATN.ME"
    
    # Даты
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365)
    
    try:
        # Скачиваем данные
        data = yf.download(
            ticker, 
            start=start_date.strftime('%Y-%m-%d'),
            end=end_date.strftime('%Y-%m-%d'),
            progress=False
        )
        
        # Преобразуем в нужный формат
        df = data.reset_index()[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
        df = df.rename(columns={
            'Date': 'date',
            'Open': 'open', 
            'High': 'high',
            'Low': 'low',
            'Close': 'close',
            'Volume': 'volume'
        })
        
        # Сортируем
        df = df.sort_values('date').reset_index(drop=True)
        
        print(f"Investing.com: получено {len(df)} записей для {ticker}")
        return df
        
    except Exception as e:
        print(f"Ошибка парсинга Investing.com: {e}")
        return pd.DataFrame()

In [92]:
df_investing = parse_investing_yndx()
print(df_investing.head())

C:\Users\baben_bakg1j1\AppData\Local\Temp\ipykernel_1056\168787197.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(

1 Failed download:
['TATN.ME']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-30 -> 2025-09-30)')


Investing.com: получено 0 записей для TATN.ME
Empty DataFrame
Columns: [(date, ), (open, TATN.ME), (high, TATN.ME), (low, TATN.ME), (close, TATN.ME), (volume, TATN.ME)]
Index: []


## 2 Успешный вариант с Мосбиржи

In [1]:
import requests
import pandas as pd
import numpy as np
import ta
from datetime import datetime, timedelta
import time

In [2]:
def calculate_daily_statistics(data_dict, window_size=30):
    """
    Расчет ежедневных статистик и индикаторов с скользящим окном
    
    Parameters:
    -----------
    data_dict : dict
        Словарь с DataFrame из функции parse_moex_data
    window_size : int
        Размер окна для скользящих статистик (по умолчанию 30 дней)
    
    Returns:
    --------
    dict
        Словарь с обогащенными DataFrame для каждого тикера
    """
    
    if not data_dict:
        print("❌ Нет данных для обработки")
        return {}
    
    print(f"=== РАСЧЕТ ЕЖЕДНЕВНЫХ СТАТИСТИК (окно {window_size} дней) ===")
    
    processed_data = {}
    
    for ticker, df in data_dict.items():
        print(f"Обрабатываем {ticker}...")
        
        if df.empty:
            print(f"  ❌ {ticker}: пустой DataFrame")
            continue
            
        # Создаем копию для обработки
        df_processed = df.copy().sort_values('date')
        
        # ===== БАЗОВЫЕ ТЕХНИЧЕСКИЕ ИНДИКАТОРЫ =====
        
        # 1. RSI
        df_processed['rsi_14'] = ta.momentum.RSIIndicator(df_processed['close'], window=14).rsi()
        
        # 2. MACD
        macd = ta.trend.MACD(df_processed['close'])
        df_processed['macd'] = macd.macd()
        df_processed['macd_signal'] = macd.macd_signal()
        df_processed['macd_histogram'] = macd.macd_diff()
        
        # 3. Скользящие средние
        df_processed['sma_20'] = ta.trend.SMAIndicator(df_processed['close'], window=20).sma_indicator()
        df_processed['ema_20'] = ta.trend.EMAIndicator(df_processed['close'], window=20).ema_indicator()
        df_processed['sma_50'] = ta.trend.SMAIndicator(df_processed['close'], window=50).sma_indicator()
        
        # 4. Bollinger Bands
        bollinger = ta.volatility.BollingerBands(df_processed['close'])
        df_processed['bb_upper'] = bollinger.bollinger_hband()
        df_processed['bb_lower'] = bollinger.bollinger_lband()
        df_processed['bb_middle'] = bollinger.bollinger_mavg()
        df_processed['bb_width'] = (df_processed['bb_upper'] - df_processed['bb_lower']) / df_processed['bb_middle']
        df_processed['bb_position'] = (df_processed['close'] - df_processed['bb_lower']) / (df_processed['bb_upper'] - df_processed['bb_lower'])
        
        # 5. Stochastic
        stoch = ta.momentum.StochasticOscillator(df_processed['high'], df_processed['low'], df_processed['close'])
        df_processed['stoch_k'] = stoch.stoch()
        df_processed['stoch_d'] = stoch.stoch_signal()
        
        # 6. Volume
        df_processed['volume_sma_20'] = ta.trend.SMAIndicator(df_processed['volume'], window=20).sma_indicator()
        df_processed['volume_ratio'] = df_processed['volume'] / df_processed['volume_sma_20']
        
        # 7. ATR и ADX
        df_processed['atr'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close']
        ).average_true_range()
        
        df_processed['adx'] = ta.trend.ADXIndicator(
            df_processed['high'], df_processed['low'], df_processed['close']
        ).adx()
        
        # ===== СКОЛЬЗЯЩИЕ СТАТИСТИКИ НА КАЖДЫЙ ДЕНЬ =====
        
        # Процентные изменения
        df_processed['price_change_1d'] = df_processed['close'].pct_change(1)
        df_processed['price_change_5d'] = df_processed['close'].pct_change(5)
        
        # Скользящая волатильность (годовая)
        df_processed['volatility_20d'] = df_processed['price_change_1d'].rolling(window=20).std() * np.sqrt(252)
        df_processed['volatility_60d'] = df_processed['price_change_1d'].rolling(window=60).std() * np.sqrt(252)
        
        # Скользящая доходность
        df_processed['return_20d'] = df_processed['close'].pct_change(20)
        df_processed['return_60d'] = df_processed['close'].pct_change(60)
        
        # Скользящий Sharpe Ratio (упрощенный)
        df_processed['sharpe_20d'] = df_processed['return_20d'] / df_processed['volatility_20d']
        df_processed['sharpe_60d'] = df_processed['return_60d'] / df_processed['volatility_60d']
        
        # Максимальная просадка в окне
        df_processed['max_drawdown_20d'] = df_processed['close'].rolling(window=20).apply(
            lambda x: (x / x.expanding().max() - 1).min(), raw=False
        )
        
        df_processed['max_drawdown_60d'] = df_processed['close'].rolling(window=60).apply(
            lambda x: (x / x.expanding().max() - 1).min(), raw=False
        )
        
        # Скользящие минимумы и максимумы
        df_processed['min_20d'] = df_processed['low'].rolling(window=20).min()
        df_processed['max_20d'] = df_processed['high'].rolling(window=20).max()
        df_processed['min_60d'] = df_processed['low'].rolling(window=60).min()
        df_processed['max_60d'] = df_processed['high'].rolling(window=60).max()
        
        # Относительные позиции
        df_processed['position_20d_range'] = (df_processed['close'] - df_processed['min_20d']) / (df_processed['max_20d'] - df_processed['min_20d'])
        df_processed['position_60d_range'] = (df_processed['close'] - df_processed['min_60d']) / (df_processed['max_60d'] - df_processed['min_60d'])
        
        # Объемные статистики
        df_processed['volume_volatility_20d'] = df_processed['volume'].rolling(window=20).std() / df_processed['volume'].rolling(window=20).mean()
        
        # Корреляция цены и объема (скользящая)
        df_processed['price_volume_corr_20d'] = df_processed['close'].rolling(window=20).corr(df_processed['volume'])
        
        # Моментum индикаторы
        df_processed['momentum_10d'] = df_processed['close'] / df_processed['close'].shift(10) - 1
        df_processed['momentum_20d'] = df_processed['close'] / df_processed['close'].shift(20) - 1
        
        # Относительная сила к рынку (если бы был индекс)
        df_processed['price_vs_sma_20'] = df_processed['close'] / df_processed['sma_20'] - 1
        df_processed['price_vs_sma_50'] = df_processed['close'] / df_processed['sma_50'] - 1
        
        # Волатильность волатильности
        df_processed['vol_of_vol_20d'] = df_processed['volatility_20d'].rolling(window=20).std()
        
        # Сезонность (день недели)
        df_processed['day_of_week'] = df_processed['date'].dt.dayofweek
        df_processed['is_monday'] = (df_processed['day_of_week'] == 0).astype(int)
        df_processed['is_friday'] = (df_processed['day_of_week'] == 4).astype(int)
        
        # Процент дней роста в окне
        df_processed['up_days_ratio_20d'] = (df_processed['price_change_1d'] > 0).rolling(window=20).mean()
        
        # Средний абсолютный дневной возврат
        df_processed['avg_abs_return_20d'] = df_processed['price_change_1d'].abs().rolling(window=20).mean()
        
        # Коэффициент отклонения от тренда
        df_processed['trend_deviation_20d'] = (df_processed['close'] - df_processed['sma_20']) / df_processed['sma_20']
        
        # Заполняем пропуски в начале (из-за скользящих окон)
        df_processed = df_processed.fillna(method='bfill').fillna(method='ffill')
        
        processed_data[ticker] = df_processed
        print(f"  ✅ {ticker}: {len(df_processed)} дней, {len(df_processed.columns)} показателей")
        
        # Показываем пример статистик для последнего дня
        last_row = df_processed.iloc[-1]
        print(f"     Последний день: RSI={last_row['rsi_14']:.1f}, "
              f"Волатильность={last_row['volatility_20d']:.1%}, "
              f"Sharpe={last_row['sharpe_20d']:.2f}")
    
    return processed_data

In [13]:
import requests
import pandas as pd
import numpy as np
import ta
from datetime import datetime, timedelta
import time

def parse_moex_data(tickers=None, days=180, interval=24):
    """
    Парсинг данных с Московской биржи с поддержкой разных интервалов
    
    Parameters:
    -----------
    tickers : list, optional
        Список тикеров для парсинга. По умолчанию основные голубые фишки
    days : int, optional
        Количество дней исторических данных (по умолчанию 180)
    interval : int, optional
        Интервал данных в минутах. Доступные значения:
        - 1: 1 минута
        - 10: 10 минут  
        - 60: 1 час
        - 24: 1 день (по умолчанию)
        - 7: 1 неделя
        - 31: 1 месяц
    
    Returns:
    --------
    dict
        Словарь с DataFrame для каждого тикера
    """
    
    # Валидные интервалы MOEX API
    valid_intervals = {
        1: "1 минута",
        10: "10 минут", 
        60: "1 час",
        24: "1 день",
        7: "1 неделя",
        31: "1 месяц"
    }
    
    if interval not in valid_intervals:
        raise ValueError(f"Неверный интервал. Допустимые значения: {list(valid_intervals.keys())}")
    
    if tickers is None:
        tickers = ['SBER', 'GAZP', 'LKOH', 'ROSN', 'YDEX', 'VTBR', 'TATN', 'GMKN']
    
    all_data = {}
    
    # Рассчитываем даты
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    print(f"=== ПАРСИНГ ДАННЫХ ЗА ПОСЛЕДНИЕ {days} ДНЕЙ ===")
    print(f"📊 Интервал: {valid_intervals[interval]}")
    
    for i, ticker in enumerate(tickers):
        print(f"({i+1}/{len(tickers)}) Получаем данные для {ticker}...")
        
        url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"
        
        params = {
            'from': start_date.strftime('%Y-%m-%d'),
            'till': end_date.strftime('%Y-%m-%d'),
            'interval': interval,
        }
        
        try:
            response = requests.get(url, params=params, timeout=15)
            data = response.json()
            
            if 'candles' in data and data['candles']['data']:
                candles = data['candles']['data']
                
                # Создаем DataFrame
                df = pd.DataFrame(candles, columns=[
                    'open', 'close', 'high', 'low', 'value', 'volume', 'begin', 'end'
                ])
                
                # Обрабатываем даты
                df['date'] = pd.to_datetime(df['begin'])
                df['ticker'] = ticker
                
                # Выбираем нужные колонки и сортируем
                df = df[['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']]
                df = df.sort_values('date').reset_index(drop=True)
                
                all_data[ticker] = df
                print(f"  ✅ {ticker}: {len(df)} записей")
                
            else:
                print(f"  ❌ {ticker}: нет данных в указанный период")
                
        except Exception as e:
            print(f"  ❌ {ticker}: ошибка - {e}")
        
        # Небольшая пауза чтобы не нагружать API
        time.sleep(0.5)
    
    print(f"\n📊 Парсинг завершен. Получено данных для {len(all_data)} тикеров")
    return all_data

stock_data = parse_moex_data(interval=10)
stock_data['GAZP']

In [21]:
stock_data = parse_moex_data(interval=10)

=== ПАРСИНГ ДАННЫХ ЗА ПОСЛЕДНИЕ 180 ДНЕЙ ===
📊 Интервал: 10 минут
(1/8) Получаем данные для SBER...
  ✅ SBER: 500 записей
(2/8) Получаем данные для GAZP...
  ✅ GAZP: 500 записей
(3/8) Получаем данные для LKOH...
  ✅ LKOH: 500 записей
(4/8) Получаем данные для ROSN...
  ✅ ROSN: 500 записей
(5/8) Получаем данные для YDEX...
  ✅ YDEX: 500 записей
(6/8) Получаем данные для VTBR...
  ✅ VTBR: 500 записей
(7/8) Получаем данные для TATN...
  ✅ TATN: 500 записей
(8/8) Получаем данные для GMKN...
  ✅ GMKN: 500 записей

📊 Парсинг завершен. Получено данных для 8 тикеров


In [23]:
stock_data['GAZP']

,date,ticker,open,high,low,close,volume
0,2025-04-08 06:50:00,GAZP,126.60,126.60,126.60,126.60,57280
1,2025-04-08 07:00:00,GAZP,126.60,127.73,126.40,127.22,2639760
2,2025-04-08 07:10:00,GAZP,127.20,127.79,126.87,127.35,1270240
3,2025-04-08 07:20:00,GAZP,127.39,127.55,127.06,127.11,637340
4,2025-04-08 07:30:00,GAZP,127.11,127.90,127.10,127.85,427150
...,...,...,...,...,...,...,...
495,2025-04-13 15:50:00,GAZP,135.95,135.95,135.95,135.95,71690
496,2025-04-13 16:00:00,GAZP,135.95,135.95,135.95,135.95,17290
497,2025-04-13 16:10:00,GAZP,135.95,135.95,135.95,135.95,24760
498,2025-04-13 16:20:00,GAZP,135.95,135.95,135.95,135.95,71070


In [190]:
def estimate_csv_size(df):
    """Оценивает примерный размер CSV файла"""
    
    # Оценка через преобразование в строку
    csv_string = df.to_csv(index=False)
    estimated_size = len(csv_string.encode('utf-8'))
    
    return estimated_size

In [198]:
estimated_size = estimate_csv_size(stock_data['SBER'])
print(f"Примерный размер CSV: {estimated_size / (1024 * 1024):.2f} МБ")

Примерный размер CSV: 0.01 МБ


In [180]:
stock_data['SBER']

,date,ticker,open,high,low,close,volume
0,2025-04-03 06:00:00,SBER,306.21,306.21,306.21,306.21,28810
1,2025-04-03 07:00:00,SBER,306.28,309.20,306.22,307.99,2023560
2,2025-04-03 08:00:00,SBER,307.95,307.99,307.15,307.42,680410
3,2025-04-03 09:00:00,SBER,307.44,307.48,304.66,305.82,2131790
4,2025-04-03 10:00:00,SBER,305.77,305.96,303.58,304.05,3348250
...,...,...,...,...,...,...,...
495,2025-05-07 07:00:00,SBER,302.51,304.21,302.51,303.64,811460
496,2025-05-07 08:00:00,SBER,303.65,304.24,303.03,303.47,799830
497,2025-05-07 09:00:00,SBER,303.43,303.47,301.23,301.42,2196640
498,2025-05-07 10:00:00,SBER,301.43,302.69,300.56,302.16,4231040


In [182]:
processed_data = calculate_daily_statistics(stock_data)

=== РАСЧЕТ ЕЖЕДНЕВНЫХ СТАТИСТИК (окно 30 дней) ===
Обрабатываем SBER...
  ✅ SBER: 500 дней, 54 показателей
     Последний день: RSI=60.0, Волатильность=7.2%, Sharpe=0.49


C:\Users\baben_bakg1j1\AppData\Local\Temp\ipykernel_1056\3001330639.py:147: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_processed = df_processed.fillna(method='bfill').fillna(method='ffill')


In [184]:
processed_data['SBER'].tail()

,date,ticker,open,high,low,close,volume,rsi_14,macd,macd_signal,...,momentum_20d,price_vs_sma_20,price_vs_sma_50,vol_of_vol_20d,day_of_week,is_monday,is_friday,up_days_ratio_20d,avg_abs_return_20d,trend_deviation_20d
495,2025-05-07 07:00:00,SBER,302.51,304.21,302.51,303.64,811460,63.163393,1.577078,1.170940,...,0.030546,0.012570,0.014815,0.007778,2,0,0,0.70,0.003717,0.012570
496,2025-05-07 08:00:00,SBER,303.65,304.24,303.03,303.47,799830,62.325383,1.582258,1.253203,...,0.029934,0.010517,0.014059,0.006943,2,0,0,0.65,0.003743,0.010517
497,2025-05-07 09:00:00,SBER,303.43,303.47,301.23,301.42,2196640,53.165257,1.404752,1.283513,...,0.018724,0.002766,0.007161,0.006277,2,0,0,0.60,0.003872,0.002766
498,2025-05-07 10:00:00,SBER,301.43,302.69,300.56,302.16,4231040,55.696518,1.308703,1.288551,...,0.024792,0.004007,0.009535,0.005483,2,0,0,0.65,0.003821,0.004007
499,2025-05-07 11:00:00,SBER,302.16,303.55,300.90,303.54,3465690,60.034485,1.328623,1.296566,...,0.035125,0.006870,0.013972,0.004805,2,0,0,0.70,0.003776,0.006870


In [138]:
processed_data['SBER'].keys()

dict_keys(['data', 'statistics'])